<a href="https://colab.research.google.com/github/mjh5153/pgp-generative-ai/blob/main/LLama_Course_Work.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# LLamindex
# embeddings-type, vector-stores
# Failures - windows - package not available ? install from command line
!pip install python-dotenv llama-index-core llama-index-llms-google-genai llama-index-embeddings-google-genai llama-index-vector-stores-faiss faiss-cpu llama-index-readers-file nest-asyncio

Custom Vector Stores - developers.llamaindex.ai/llamaparse/cloud-index-v2/

In [7]:
import nest_asyncio
nest_asyncio.apply()
# main method ? need await : directly -> await directly (that's what we are doing)
from dotenv import load_dotenv
from llama_index.core import (
    # read file from directory
    SimpleDirectoryReader,
    VectorStoreIndex,
    StorageContext,
    PromptTemplate
) # parent level

#3. LLM level

from llama_index.core.node_parser import SentenceSplitter # llama doesn't have textsplitter
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.vector_stores.faiss import FaissVectorStore # vector store

import faiss
import os # operating system

# load api keys
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    try:
        from google.colab import userdata
        GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
    except Exception:
        pass

if not GOOGLE_API_KEY:
    raise ValueError("No API key was found. Please either create a .env file with GOOGLE_API_KEY=your_key or set it in Google Colab Secrets (the key icon on the left panel) with the name GOOGLE_API_KEY.")

# load the document ==> text document (data.txt)
documents = SimpleDirectoryReader(input_files=["data.txt"], encoding="utf-8").load_data()

# Text splitter
text_splitter = SentenceSplitter(chunk_size=500, chunk_overlap=50) # GPU ideal, power consumption

# embedding
# Embeddings ==> using google's gemini-embedding-001 - Mofication model - GoogleGenAIEmbedding
embeddings = GoogleGenAIEmbedding(model_name="models/gemini-embedding-001", api_key=GOOGLE_API_KEY)

# Vector store - small mod from LangGraph
dim = len(embeddings.get_text_embedding("hello"))
vector_store = FaissVectorStore(faiss_index=faiss.IndexFlatL2(dim))
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex.from_documents(documents,
    storage_context=storage_context,
    embed_model=embeddings,
    transformations=[text_splitter])

# llm model
llm = GoogleGenAI(model="models/gemini-3.8-flash", api_key=GOOGLE_API_KEY)

# prompt
prompt = PromptTemplate(
    "Answer the question based only on the context:\n\n{context_str}\n\nQuestion: {query_str}"
)

# rag_pipline
query_engine = index.as_query_engine(
    llm=llm,
    text_qa_template=prompt,
    similarity_top_k=4,
)

query = "What is this document all about?"

response = await query_engine.aquery(query) # circuit breaker and retry mechanisms with
# try catch blocks
print(response)

Based on the context, this document is about Lewis Carroll's 1865 children's novel ***Alice's Adventures in Wonderland***. 

Specifically, it details:
* An overview of the book, its plot, literary nonsense genre, and original illustrations by John Tenniel.
* Its positive reception, cultural impact, lasting legacy, translations, adaptations, and sequels.
* The origins and conception of the story during an 1862 river trip with the Liddell children, Carroll's relationship with the children (especially Alice Liddell), and how the story was originally created and written down for her.


In [9]:
#
dir(index.as_query_engine)

['__call__',
 '__class__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__func__',
 '__ge__',
 '__get__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__self__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__']

In [10]:
# Next Module - Phidata - open source python framework for building the LLM powered agents and AI application

### it can:
# reason
# use tools
# access memory
# work with data
# work with api
## in a structured way

## Defines AI agents with clear roles
## give them tools (api, db, code executions, web searches, etc... )
## add memory and context
## orchestrate workflowsand multi-step reasoning
## deploy the agents as services or app

## Reasons for choosing phidata
### Simplicity
### declarative configurations
### productive readiness

### Note: Avoid for enterprise level deployment - application development, because it is meant for simplicity

### Sample Program - Web Search program

# no code or low code

# Not for Enterprise - Use LangGraph: Agent Single/Multi

# Simple - Phidata - PRoduction ( not enterprise - poc, simple app)
# CrewAI -> low code ( performance hit )
# Microsoft - Autogen -- Tradeoff - Performance issues

# Can compare with Google Trends - explore - LangGraph, crewai, phidata, autogen

In [12]:
!pip install phidata duckduckgo-search ddgs yfinance

/usr/lib/python3.13/pathlib/_local.py:277: RuntimeWarning: coroutine 'prepare_chat_params' was never awaited
  @property


In [20]:
# Search engine, AI model, Agent
import os
from dotenv import load_dotenv

from phi.agent import Agent
from phi.model.openai import OpenAIChat
from phi.tools.duckduckgo import DuckDuckGo
from phi.tools.yfinance import YFinanceTools

load_dotenv()
# Correct the environment variable name to OPENAI_API_KEY
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    try:
        from google.colab import userdata
        # Fallback to OPENAI_API_KEY or OPEN_API_KEY from Colab Secrets
        OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    except Exception:
        try:
            from google.colab import userdata
            OPENAI_API_KEY = userdata.get("OPEN_API_KEY")
        except Exception:
            pass

# Ensure the environment variable is globally visible to Phidata / OpenAI clients
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
else:
    raise ValueError("No OpenAI API key was found. Please set OPENAI_API_KEY in your secrets or .env file.")

# step 2 - create agent
web_agent = Agent(
    name="Web Agent",
    # model type i.e GPT-4o, explicitly passing the retrieved API key
    model=OpenAIChat(id="gpt-4o", api_key=OPENAI_API_KEY),
    tools=[DuckDuckGo()],
    instructions=["Always include the sources"],
    show_tool_calls=True,
    markdown=True
)

# Can define new agents i.e. finance_agent
finance_agent = Agent(
    name="Finance Agent",
    # model type i.e GPT-4o, explicitly passing the retrieved API key
    model=OpenAIChat(id="gpt-4o", api_key=OPENAI_API_KEY),
    tools=[YFinanceTools(stock_price=True, analyst_recommendations=True, stock_fundamentals=True)],
    instructions=["Use tables to show data where possible."],
    show_tool_calls=True,
    markdown=True
)

# define agent_team
agent_team = Agent(
    team=[web_agent, finance_agent],
    instructions=["Always include the sources", "Fetch the latest analyst recommendations and stock prices. Always show results in tables."],
    show_tool_calls=True,
    markdown=True
)

# Completed the cut-off prompt
agent_team.print_response("What is the stock price of Apple and NVIDIA?")

# Pattern: AI, Agents, SWE

# Upload own custom agents - community - LangChain Docs - Integrations - tools and toolkits - Contributing to integrations - contribute code - LangGraph,
# langgraph+community agents  - langgraph-quick-start.py.md -- need to follow community rules
# own packages - pypi i.e pypi.org/projects/finance/ -- python package hosting

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                              ┃
┃ What is the stock price of Apple and NVIDIA?                                 ┃
┃                                                                              ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (6.6s) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                              ┃
┃ Running:                                                                     ┃
┃                                                                              ┃
┃  • transfer_task_to_finance_agent(task_description=Fetch the current stock   ┃
┃    price for Apple Inc (AAPL)., expected_output=The current stock price of   ┃
┃    Apple Inc (AAPL)., additional_information=)                               ┃
┃  • transfer_task_to_financ

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag